# Input

In [1]:
target_column = "RecommendHiring"
%load_ext autoreload
%autoreload 2
VIDEOS_FOLDER = "./data/raw/videos"
import importlib.util
from pathlib import Path
import os

# Constants

In [2]:
import os
import numpy as np
from sklearn.metrics import make_scorer

SCRIPT_DIR = os.path.join(os.getcwd(),)

SAVED_MODELS_PATH="./../../model_artifacts"

DROPPED_LEXICAL_COLUMNS = [
    "Swear",
    "Numbers",
    "Inhibition",
    "Preceptual",
    "Anxiety",
    "Anger",
    "Sadness",
    "Work",
    "Articles",
    "Verbs",
    "Adverbs",
    "Prepositions",
    "Conjunctions",
    "Negations",
]

DROPPED_PROSODIC_FEATURES = []

facial_features = [
    "average_inner_brow_height",
    "average_outer_brow_height",
    "eye_open",
    "inner_lip_height",
    "lip_corner_distance",
    "outer_lip_height",
    "smile",
    "pitch",
    "roll",
    "yaw",
]

stats = ["max", "median", "min", "std"]
DROPPED_FACIAL_FEATURES = [
    f"{feature}_{stat}" for feature in facial_features for stat in stats
]



ALREADY_NORMALIZED_FEATURES = [
    "average_outer_brow_height_mean",
    "average_inner_brow_height_mean",
    "eye_open_mean",
    "inner_lip_height_mean",
    "inner_lip_height_mean",
    "lip_corner_distance_mean",
    "average_outer_brow_height_std",
    "average_inner_brow_height_std",
    "eye_open_std",
    "outer_lip_height_std",
    "inner_lip_height_std",
    "lip_corner_distance_std",
    "average_outer_brow_height_min",
    "average_inner_brow_height_min",
    "eye_open_min",
    "outer_lip_height_min",
    "inner_lip_height_min",
    "lip_corner_distance_min",
    "average_outer_brow_height_max",
    "average_inner_brow_height_max",
    "eye_open_max",
    "outer_lip_height_max",
    "inner_lip_height_max",
    "lip_corner_distance_max",
    "average_outer_brow_height_median",
    "average_inner_brow_height_median",
    "eye_open_median",
    "outer_lip_height_median",
    "inner_lip_height_median",
    "lip_corner_distance_median",
]  # these are already in [0, 1]

MUST_KEEP_FEATURES = [
    "pause_duration_avg",
    "average_outer_brow_height_mean",
    "average_inner_brow_height_mean",
    "outer_lip_height_mean",
    "Duration/Filler Words",
]


GROUPS_COLUMN = "cleaned_ids"
INDEX_COLUMN = "participant_id"


def pearson_corr(y_true, y_pred):
    return np.corrcoef(y_true, y_pred)[0, 1]


SCORING_METRICS = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "pearson": make_scorer(pearson_corr),  # Pearson Correlation Coefficient
}


MUST_KEEP_FEATURES = [
    # "pause_duration_avg",
    # "average_outer_brow_height_mean",
    # "average_inner_brow_height_mean",
    # "outer_lip_height_mean",
    "Duration/Filler Words",
]

PIPELINE_PARAMS = {'feature_selection__estimator__alpha': 0.057376790661083456, 'svr__C': 0.655379988356498, 'svr__gamma': 0.02784736494309893, 'svr__epsilon': 0.2617249201838037, 'svr__kernel': 'rbf'}
HYPERPARAMETER_TUNING_ENABLED = False

# Data Preprocessing

## Import Datasets

In [3]:
import pandas as pd
import os
from hireverse.utils.utils import * 
features_df = pd.read_csv(BASE_DIR+"/data/external/add.csv")
features_df = features_df.set_index("participant_id")

labels_df = pd.read_csv(
    os.path.join(BASE_DIR+"/data/external/turker_scores_full_interview.csv")
)
labels_df = labels_df.set_index("Participant")
labels_df = labels_df.loc[labels_df["Worker"] == "AGGR"]

features_df.index = features_df.index.str.lower()
labels_df.index = labels_df.index.str.lower()
indexed_combined_df = features_df.join(labels_df[[target_column]], how="left")
combined_df = indexed_combined_df.reset_index(drop=True)

# Model

## Split Data

In [4]:
X = combined_df.drop(columns=[target_column, GROUPS_COLUMN])
y = combined_df[target_column]

In [5]:
X

,f0_mean,f0_min,f0_max,f0_range,f0_sd,intensity_mean,intensity_min,intensity_max,intensity_range,intensity_sd,...,Work,Swear,Articles,Verbs,Adverbs,Prepositions,Conjunctions,Negations,Quantifiers,Numbers
0,138.831117,75.345596,599.146951,523.801355,67.085301,-17.989480,-36.430290,0.000000e+00,36.430290,7.597978,...,5,0,34,64,43,55,35,0,1,7
1,194.531641,74.930338,599.882668,524.952330,156.844796,-22.336040,-39.191290,0.000000e+00,39.191290,8.249971,...,1,0,53,184,151,112,83,1,10,11
2,145.412942,74.945369,597.352468,522.407099,67.823642,-19.039644,-33.233150,4.770000e-07,33.233150,6.990975,...,8,0,51,88,58,84,42,0,1,3
3,143.177152,75.352185,599.339791,523.987606,94.379153,-16.840088,-28.830551,0.000000e+00,28.830551,5.876480,...,4,0,27,57,25,40,48,0,1,1
4,110.735157,74.990604,599.083559,524.092955,46.303907,-19.337439,-29.820957,0.000000e+00,29.820957,4.713671,...,10,0,59,148,43,99,93,1,2,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,242.085132,76.132898,598.509756,522.376859,50.947813,-15.526934,-30.096924,9.540000e-07,30.096924,6.849392,...,1,0,20,60,52,54,26,1,1,4
133,226.371853,103.742483,594.074757,490.332274,42.869122,-31.422302,-47.013687,0.000000e+00,47.013687,7.368369,...,5,0,53,123,85,88,58,5,6,10
134,115.487274,74.944032,591.304356,516.360324,59.665370,-17.641912,-32.484737,0.000000e+00,32.484737,6.131927,...,5,0,62,199,160,114,84,2,1,12
135,154.635209,74.937003,598.083526,523.146524,73.160382,-20.179580,-36.918100,0.000000e+00,36.918100,8.547647,...,3,0,47,91,51,49,50,0,3,3


## Pipeline Creation

In [6]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel, SelectPercentile, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import GroupKFold, cross_val_score, cross_validate
import sys

groups_column = combined_df[GROUPS_COLUMN].astype(str).values

preprocessor = ColumnTransformer(
    [
        ('dropper', 'drop', DROPPED_FACIAL_FEATURES + 
                            DROPPED_LEXICAL_COLUMNS + 
                            DROPPED_PROSODIC_FEATURES)
    ],
    remainder='passthrough'
)

ridge_lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("feature_selection", SelectFromModel(Lasso(alpha=0.05737679, max_iter=10000))), 
    ("ridge", Ridge(alpha=1.0, solver="svd")), 
])
ridge_pval_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    # Using percentile=25 as a heuristic without tuning
    ("feature_selection_pval", SelectPercentile(f_regression, percentile=25)), # <--- Using percentile=25
    ("ridge", Ridge(alpha=1.0, solver="svd")), # Ridge alpha might also need tuning
])
svr_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ("imputer", SimpleImputer(strategy="mean")),  # NaN imputation
        ("scaler", StandardScaler()),
        ("feature_selection", SelectFromModel(estimator=Lasso(max_iter=50000))),
        ("svr", SVR(kernel="rbf")),
    ]
)

lasso_pipeline = Pipeline(
    [
        ('preprocessor', preprocessor), # Keep this if it's part of your overall data preparation
        ("imputer", SimpleImputer(strategy="mean")),  # NaN imputation
        ("scaler", StandardScaler()),
        ("lasso_model", Lasso(max_iter=50000, random_state=42)), # Lasso does its own feature selection
    ]
)

unfitted_pipeline = svr_pipeline

## Hyperparameter Tuning

In [7]:
import optuna
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, cross_val_score
import numpy as np

def objective(trial):
    pipeline_clone = clone(unfitted_pipeline)  # Clone pipeline for thread safety
    
    params = {
        "feature_selection__estimator__alpha": trial.suggest_float(
        "feature_selection__estimator__alpha", 1e-4, 0.1, log=True  # Adjusted lower bound
        ),
        "svr__C": trial.suggest_float("svr__C", 0.01, 100, log=True),
        "svr__gamma": trial.suggest_float("svr__gamma", 1e-3, 1e1, log=True),
        "svr__epsilon": trial.suggest_float("svr__epsilon", 0.01, 0.5),
        "svr__kernel": trial.suggest_categorical("svr__kernel", ["rbf"
                                                                #  , "poly"
                                                                 ]),
    }
    
    # if params["svr__kernel"] == "poly":
    #     params["svr__degree"] = trial.suggest_int("svr__degree", 2, 3)  # Reduced from 5
    #     params["svr__coef0"] = trial.suggest_float("svr__coef0", 0.0, 0.5)  # Narrower range
        
    pipeline_clone.set_params(**params)
    
    mc_cv_tuning = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=42)
    
    scores = cross_val_score(
        pipeline_clone, X, y, cv=mc_cv_tuning, groups=groups_column, n_jobs=1
    )

    return np.mean(scores)

if HYPERPARAMETER_TUNING_ENABLED:
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=100, n_jobs=-1, timeout=4*60)
    
    print("Best hyperparameters:", study.best_params)
    print(f"Best R² score: {study.best_value:.4f}")

/Users/bassel27/personal_projects/hireverse/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Store Trained Model

In [8]:
from sklearn import clone

if HYPERPARAMETER_TUNING_ENABLED:
    unfitted_pipeline.set_params(**study.best_params)
else:
    unfitted_pipeline.set_params(**PIPELINE_PARAMS)
fitted_pipeline = clone(unfitted_pipeline)
fitted_pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('dropper', 'drop',
                                                  ['average_inner_brow_height_max',
                                                   'average_inner_brow_height_median',
                                                   'average_inner_brow_height_min',
                                                   'average_inner_brow_height_std',
                                                   'average_outer_brow_height_max',
                                                   'average_outer_brow_height_median',
                                                   'average_outer_brow_height_min',
                                                   'average_oute...
                                                   'outer_lip_height_std',
                                                   'smile_max', 'smile_median',
                                                   'smile_min', 'smile_std',
                                                   'pitch_max', 'pitch_median', ...])])),
                ('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('feature_selection',
                 SelectFromModel(estimator=Lasso(alpha=0.057376790661083456,
                                                 max_iter=50000))),
                ('svr',
                 SVR(C=0.655379988356498, epsilon=0.2617249201838037,
                     gamma=0.02784736494309893))])

## Feature Selection Results

In [9]:
preprocessor = fitted_pipeline.named_steps['preprocessor']
feature_names = preprocessor.get_feature_names_out()    # after preprocessing

feature_selector = fitted_pipeline.named_steps['feature_selection']
selected_mask = feature_selector.get_support()

selected_features = feature_names[selected_mask]
unselected_features = feature_names[~selected_mask]

print(f"Number of Selected features ({len(selected_features)}):")
print(f"Selected features: {selected_features}")
print(f"Unselected features: {unselected_features}")

Number of Selected features (12):
Selected features: ['remainder__intensity_mean' 'remainder__f3_sd' 'remainder__f2_f1_mean'
 'remainder__percent_unvoiced' 'remainder__percent_breaks'
 'remainder__inner_lip_height_mean' 'remainder__lip_corner_distance_mean'
 'remainder__smile_mean' 'remainder__roll_mean'
 'remainder__Duration/Total Words' 'remainder__They'
 'remainder__Cognitive']
Unselected features: ['remainder__f0_mean' 'remainder__f0_min' 'remainder__f0_max'
 'remainder__f0_range' 'remainder__f0_sd' 'remainder__intensity_min'
 'remainder__intensity_max' 'remainder__intensity_range'
 'remainder__intensity_sd' 'remainder__f1_mean' 'remainder__f1_sd'
 'remainder__f2_mean' 'remainder__f2_sd' 'remainder__f3_mean'
 'remainder__f3_f1_mean' 'remainder__f2_f1_sd' 'remainder__f3_f1_sd'
 'remainder__jitter' 'remainder__shimmer' 'remainder__pause_duration_max'
 'remainder__pause_duration_avg' 'remainder__duration'
 'remainder__average_outer_brow_height_mean'
 'remainder__average_inner_brow_hei

# Monte Carlo Cross Validation

In [10]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import r2_score

scoring = {
    "r2": make_scorer(r2_score),
    "pearson": make_scorer(pearson_corr)
}

results = cross_validate(
    unfitted_pipeline,
    X,
    y,
    cv=GroupShuffleSplit(n_splits=1000, test_size=0.2, random_state=42),
    groups=groups_column,
    scoring=scoring,
    n_jobs=-1,
)

r2_scores = results["test_r2"]
pearson_scores = results["test_pearson"]
# Report the average performance and variability
avg_r2_score = np.mean(r2_scores)
print(
    f"Mean R² Score: {avg_r2_score:.2f} (±{np.std(r2_scores):.2f})"
)
avg_perason_score = np.mean(pearson_scores)
print(
    f"Mean Pearson Correlation: {avg_perason_score:.2f} (±{np.std(pearson_scores):.2f})"
)

# Mean R² Score: 0.20 (±0.23)
# Mean Pearson Correlation: 0.53 (±0.16)

Mean R² Score: 0.17 (±0.22)
Mean Pearson Correlation: 0.50 (±0.15)


# Save the Model

In [11]:
from joblib import dump, load

dump(fitted_pipeline, os.path.join(BASE_DIR,'model_artifacts',f'{target_column}.joblib'))

['/Users/bassel27/personal_projects/hireverse/model_artifacts/RecommendHiring.joblib']